# PF4 Associations from the GWAS Catalog

This notebook queries the GWAS Catalog API to retrieve known SNP–trait associations related to the PF4 gene. Two complementary strategies are used: a gene-based query to identify associations mapped to PF4, and a variant-based query to search for associations of selected priority PF4 SNPs from NCBI dbSNP. The results are normalized, combined, and annotated with variant-level information from the PF4 variant dataset.

In [39]:
import requests
import pandas as pd
import json
import time
from pathlib import Path

pd.set_option("display.max_columns", None)

In [40]:
region_path = Path("../data/interim/ensembl/region.json")
variants_path = Path("../data/interim/ncbi/variants.csv")

out_dir = Path("../data/interim/gwas_catalog")
out_dir.mkdir(parents=True, exist_ok=True)

out_gene_csv = out_dir / "gene_associations.csv"
out_variant_csv = out_dir / "variant_associations.csv"
out_combined_csv = out_dir / "associations.csv"

In [41]:
with open(region_path, "r") as f:
    pf4_region = json.load(f)

variants_df = pd.read_csv(variants_path)

pf4_region

{'gene_symbol': 'PF4',
 'ensembl_gene_id': 'ENSG00000163737',
 'display_name': 'PF4',
 'species': 'homo_sapiens',
 'chromosome': '4',
 'gene_start': 73980811,
 'gene_end': 73982027,
 'strand': -1,
 'assembly_name': 'GRCh38',
 'biotype': 'protein_coding',
 'source': 'Ensembl REST lookup/symbol',
 'flanking_bp': 50000,
 'region_start': 73930811,
 'region_end': 74032027}

In [42]:
gene_symbol = pf4_region["gene_symbol"]
chromosome = pf4_region["chromosome"]
region_start = pf4_region["region_start"]
region_end = pf4_region["region_end"]
assembly = pf4_region["assembly_name"]

region_string = f"chr{chromosome}:{region_start}-{region_end}"

print("Gene:", gene_symbol)
print("Region:", region_string)
print("Assembly:", assembly)
print("Number of PF4 variants:", variants_df.shape[0])

Gene: PF4
Region: chr4:73930811-74032027
Assembly: GRCh38
Number of PF4 variants: 2030


In [43]:
priority_variants = variants_df[
    variants_df["Priority"].isin(["Primary", "Secondary"])
]["rsID"].dropna().unique().tolist()

len(priority_variants)

176

In [44]:
BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

HEADERS = {
    "Accept": "application/json"
}

REQUEST_DELAY = 0.1

In [45]:
def gwas_api_request(endpoint, params=None):
    """
    Send a GET request to the GWAS Catalog REST API V2.
    """
    url = f"{BASE_URL}{endpoint}"

    response = requests.get(
        url,
        params=params,
        headers=HEADERS,
        timeout=30
    )

    if not response.ok:
        print("Request failed")
        print("URL:", response.url)
        print("Status code:", response.status_code)
        print("Response:", response.text[:500])
        response.raise_for_status()

    return response.json()

In [46]:
def fetch_paginated(endpoint, params=None, embedded_key=None, max_pages=None):
    """
    Fetch all pages from a paginated GWAS Catalog endpoint.
    """
    params = params.copy() if params else {}
    params.setdefault("size", 200)

    all_records = []
    page = 0
    total_pages = 1

    while page < total_pages:
        params["page"] = page

        data = gwas_api_request(endpoint, params=params)

        if "_embedded" not in data:
            break

        embedded = data["_embedded"]

        if embedded_key is None:
            embedded_key = list(embedded.keys())[0]

        records = embedded.get(embedded_key, [])
        all_records.extend(records)

        page_info = data.get("page", {})
        total_pages = page_info.get("totalPages", 1)

        page += 1
        time.sleep(REQUEST_DELAY)

        if max_pages is not None and page >= max_pages:
            break

    return all_records

In [47]:
def extract_trait_names(efo_traits):
    if not isinstance(efo_traits, list):
        return None

    traits = []
    for trait in efo_traits:
        if isinstance(trait, dict):
            name = trait.get("efo_trait")
            if name:
                traits.append(name)

    return "; ".join(traits) if traits else None


def extract_trait_ids(efo_traits):
    if not isinstance(efo_traits, list):
        return None

    ids = []
    for trait in efo_traits:
        if isinstance(trait, dict):
            trait_id = (
                trait.get("efo_id")
                or trait.get("short_form")
                or trait.get("uri")
                or trait.get("id")
            )
            if trait_id:
                ids.append(trait_id)

    return "; ".join(ids) if ids else None


def extract_rsid_and_allele(snp_effect_allele):
    if not isinstance(snp_effect_allele, list) or len(snp_effect_allele) == 0:
        return None, None

    value = snp_effect_allele[0]

    if not isinstance(value, str):
        return None, None

    if "-" in value:
        rsid, allele = value.split("-", 1)
        return rsid, allele

    return value, None


def list_to_string(value):
    if isinstance(value, list):
        return "; ".join(str(v) for v in value)
    return value

In [48]:
def normalize_association(association, query_source=None, query_value=None):
    rsid, risk_allele = extract_rsid_and_allele(
        association.get("snp_effect_allele")
    )

    return {
        "query_source": query_source,
        "query_value": query_value,
        "association_id": association.get("association_id"),
        "rsID": rsid,
        "risk_allele": risk_allele,
        "p_value": association.get("p_value"),
        "risk_frequency": association.get("risk_frequency"),
        "mapped_genes": list_to_string(association.get("mapped_genes")),
        "trait_name": extract_trait_names(association.get("efo_traits")),
        "trait_id": extract_trait_ids(association.get("efo_traits")),
        "accession_id": association.get("accession_id"),
        "pubmed_id": association.get("pubmed_id"),
        "first_author": association.get("first_author"),
    }

## Gene-based associations

In [49]:
gene_params = {
    "mapped_gene": gene_symbol,
    "extended_geneset": "false",
    "sort": "p_value",
    "direction": "asc",
    "size": 200
}

gene_associations_raw = fetch_paginated(
    endpoint="/v2/associations",
    params=gene_params,
    embedded_key="associations"
)

len(gene_associations_raw)

107

In [50]:
gene_associations_df = pd.DataFrame([
    normalize_association(
        assoc,
        query_source="mapped_gene",
        query_value=gene_symbol
    )
    for assoc in gene_associations_raw
])

gene_associations_df.head()

,query_source,query_value,association_id,rsID,risk_allele,p_value,risk_frequency,mapped_genes,trait_name,trait_id,accession_id,pubmed_id,first_author
0,mapped_gene,PF4,164774235,rs74499711,?,0.000000e+00,NR,PF4; CXCL1P1,CXCL1/CXCL3 protein level ratio in blood,OBA_2055094,GCST90314342,38412862,Suhre K
1,mapped_gene,PF4,104535614,rs11574450,A,0.000000e+00,0.95,PF4,growth-regulated alpha protein measurement,EFO_0008146,GCST90247816,34648354,Pietzner M
2,mapped_gene,PF4,164732792,rs2367442,?,4.000000e-232,0.2187,PF4; CXCL1P1,CXCL1 measurement,EFO_0010777,GCST90428404,39794498,Konieczny MJ
3,mapped_gene,PF4,188226556,rs614822,A,1.000000e-101,0.88635332,PF4; CXCL1P1,growth-regulated alpha protein measurement,EFO_0008146,GCST90468930,39789286,Loya H
4,mapped_gene,PF4,188226471,rs138657097,A,9.000000e-39,0.00171965,PF4; CXCL1P1,growth-regulated alpha protein measurement,EFO_0008146,GCST90468930,39789286,Loya H


## Variant-based associations

In [51]:
variant_association_rows = []

for rsid in priority_variants:
    params = {
        "rs_id": rsid,
        "sort": "p_value",
        "direction": "asc",
        "size": 200
    }

    records = fetch_paginated(
        endpoint="/v2/associations",
        params=params,
        embedded_key="associations"
    )

    for assoc in records:
        variant_association_rows.append(
            normalize_association(
                assoc,
                query_source="rsID",
                query_value=rsid
            )
        )

    time.sleep(REQUEST_DELAY)

variant_associations_df = pd.DataFrame(variant_association_rows)

variant_associations_df.shape

(88, 13)

## Combine gene-based and variant-based associations

In [52]:
combined_df = pd.concat(
    [gene_associations_df, variant_associations_df],
    ignore_index=True
)

combined_df = combined_df.drop_duplicates(subset=["association_id"])

combined_df["p_value"] = pd.to_numeric(combined_df["p_value"], errors="coerce")

combined_df = combined_df.sort_values("p_value", ascending=True)

combined_df.shape

(107, 13)

In [53]:
selected_trait_ids = {
    "MONDO_0005252",  # Heart failure
    "MONDO_0005010",  # Coronary artery disease
    "MONDO_0005068",  # Myocardial infarction
}

In [54]:
def is_selected_trait(trait_ids):
    if pd.isna(trait_ids):
        return False

    ids = [tid.strip() for tid in str(trait_ids).split(";")]

    return any(tid in selected_trait_ids for tid in ids)


combined_df["is_selected_trait"] = combined_df["trait_id"].apply(
    is_selected_trait
)

In [55]:
variant_priority_cols = [
    "rsID",
    "MAX_freq",
    "Priority",
    "Variant_type",
    "Alleles_REF_ALT",
    "CHRPOS_GRCh38",
    "Functional_consequence",
    "Clinical_significance"
]

variant_metadata = variants_df[variant_priority_cols].drop_duplicates()

combined_annotated_df = combined_df.merge(
    variant_metadata,
    on="rsID",
    how="left"
)

combined_annotated_df.head()

,query_source,query_value,association_id,rsID,risk_allele,p_value,risk_frequency,mapped_genes,trait_name,trait_id,accession_id,pubmed_id,first_author,is_selected_trait,MAX_freq,Priority,Variant_type,Alleles_REF_ALT,CHRPOS_GRCh38,Functional_consequence,Clinical_significance
0,mapped_gene,PF4,164774235,rs74499711,?,0.000000e+00,NR,PF4; CXCL1P1,CXCL1/CXCL3 protein level ratio in blood,OBA_2055094,GCST90314342,38412862,Suhre K,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mapped_gene,PF4,104535614,rs11574450,A,0.000000e+00,0.95,PF4,growth-regulated alpha protein measurement,EFO_0008146,GCST90247816,34648354,Pietzner M,False,0.115542,Primary,snv,A>C,4:73981394,intron_variant,NaN
2,mapped_gene,PF4,164732792,rs2367442,?,4.000000e-232,0.2187,PF4; CXCL1P1,CXCL1 measurement,EFO_0010777,GCST90428404,39794498,Konieczny MJ,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,mapped_gene,PF4,188226556,rs614822,A,1.000000e-101,0.88635332,PF4; CXCL1P1,growth-regulated alpha protein measurement,EFO_0008146,GCST90468930,39789286,Loya H,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mapped_gene,PF4,188226471,rs138657097,A,9.000000e-39,0.00171965,PF4; CXCL1P1,growth-regulated alpha protein measurement,EFO_0008146,GCST90468930,39789286,Loya H,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [56]:
gene_associations_df.to_csv(out_gene_csv, index=False)
variant_associations_df.to_csv(out_variant_csv, index=False)
combined_annotated_df.to_csv(out_combined_csv, index=False)

In [57]:
# Summary
summary = {
    "gene_query_associations": len(gene_associations_df),
    "variant_query_associations": len(variant_associations_df),
    "combined_associations": len(combined_annotated_df),
    "selected_trait_associations": int(
        combined_df["is_selected_trait"].sum()
    ),
    "unique_rsIDs_with_associations": combined_annotated_df["rsID"].nunique(),
}

summary

{'gene_query_associations': 107,
 'variant_query_associations': 88,
 'combined_associations': 107,
 'selected_trait_associations': 0,
 'unique_rsIDs_with_associations': 23}